# D2 · Calibración espectral

**Spec:** [`docs/spec_D2_codex_spectral_calibration.md`](../docs/spec_D2_codex_spectral_calibration.md)  |  **Bloque:** D · Método  |  **Run de este set:** `ROXs42Bb_realigned`

Calibra los 6 métodos y la primaria (Δλ, escala de flujo, continuo, presupuesto de error) y produce los espectros definitivos, con `spec_final_object.fits` como canónico.

| | |
|---|---|
| **Entrada** | Los 6 métodos + la primaria de C4 + M3 |
| **Salida (QC/productos)** | `stages/stage_x11_qc.json`, `spec_final_object.fits`, `spec_calibrated_<método>_object.fits` (6), `spec_calibrated_psffit_star.fits` |
| **Consume aguas abajo** | E1, E3, G2 |


## Qué hace D2 y qué integramos

**Los espectros definitivos son el entregable de esta etapa**, no solo la entrada de E1: el compañero por los **6 métodos** (misma rejilla, superponibles) y la **primaria** (`spec_calibrated_psffit_star.fits`), todos con unidad declarada (`BUNIT`) y error total. El canónico se copia además a `spec_final_object.fits`.

D2 convierte el espectro canónico (psffit) en el **producto científico final**: λ corregida y en marco declarado, flujo en escala validada, continuo por dos vías, y un **error total con presupuesto de sistemáticos explícito**. **Aplica factores medidos aguas arriba** (trazables al QC que los midió) — no mide nada nuevo.

- **λ:** Δλ = −0.052 Å (de A4/M1), marco final **baricéntrico**.
- **Flujo:** escala = 1.0 (M3 factor 0.957 vs Gaia DR3, estado **green**: se aplica escala 1.0 porque el factor es consistente con 1 dentro de su error).
- **Continuo:** running-median y polinomio; su diferencia es el término `sys_continuum`.
- **Error:** `stat` (empírico, M5 rojo) + sistemáticos (flujo-cal, psf, cielo, telúrico, continuo). El total está **dominado por el stat**.

**Diagnóstico del 'continuo rojo inestable'** ([`docs/d2_red_continuum_diagnosis.md`](../docs/d2_red_continuum_diagnosis.md)): son 3 cosas reales (señal de enana fría + sistemático de nivel inter-método + rigidez del polinomio), **no** un defecto de PSF.

**El chequeo de continuo estable (`v3_continuum_stable`)** pregunta, en físico: *¿el continuo que llamamos «del compañero» es suyo, o es lo que quedó del halo de la primaria?* Se responde comparando dos métodos de extracción independientes — abajo, en el Plot 1, está explicado en detalle. Para este objeto: 0.579 de los canales concuerdan (umbral 0.9), así que el chequeo **False** = ok. Desde 2026-07-11 la métrica es la **control-referenciada** (consistente con el t control-centrado de D1) y el producto final lleva la columna `cont_runmed_biasref` para G3. **Lo que queda es un sistemático cromático genuino, y afecta a la FORMA del continuo (G3), no a la línea Hα ni al límite de Ṁ (E1/E3).**


## Cómo ejecutar de forma independiente

```bash
conda activate MUSE               # kernel/env con astropy + musepipe
export RUN=ROXs42Bb_realigned   # el run de este objeto
cd MUSE-accretion-pipeline                    # raíz del repo
bash scripts/stage_x11_calibrate.sh --run-id $RUN
```

Ligero–moderado.

La celda de abajo hace lo mismo desde el notebook (guardada por `RUN`).


In [ ]:
import os, sys
# Localiza la raíz del repo ascendiendo hasta encontrar `musepipe/` (robusto a
# la profundidad: funciona con el cwd en notebooks/<obj>/, en notebooks/ o en la
# raíz). Añade la raíz (para `import musepipe`) y notebooks/ (para `_nbcommon`).
_d = os.getcwd()
while _d != os.path.dirname(_d):
    if os.path.isdir(os.path.join(_d, 'musepipe')) and os.path.isdir(os.path.join(_d, 'notebooks')):
        break
    _d = os.path.dirname(_d)
_root = _d
for _p in (_root, os.path.join(_root, 'notebooks')):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
RUN_ID = nb.resolve_run_id('ROXs42Bb_realigned')
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))
print('QC   =', nb.provenance_line('stages/stage_x11_qc.json', RUN_ID))


## Ejecutar o auditar


In [ ]:
RUN = False   # -> True para RE-EJECUTAR esta etapa (regenera su QC)

if RUN:
    cmd = 'bash scripts/stage_x11_calibrate.sh --run-id $RUN'.replace('$RUN', RUN_ID)
    print('ejecutando:', cmd)
    import subprocess
    subprocess.run(cmd, shell=True, cwd=str(nb.project_root()), check=True)
else:
    print('Modo auditoría (RUN=False): se carga el QC existente abajo.')


## QC / resultados


In [ ]:
qc = nb.load_qc_optional('stages/stage_x11_qc.json', RUN_ID)
nb.show(qc, keys=['canonical_method', 'scale_factor', 'v3_continuum_stable.ok', 'fraction_channels_methods_agree', 'control_referenced'], title='D2')


## Resultados que llevaron a la conclusión

Calibración aplicada + la referenciación de continuo (antes/después) del `stage_x11_qc.json`.


In [ ]:
if qc is None:
    print('(evidencia omitida: la etapa no se ha ejecutado para esta cadena)')
else:
    with nb.evidence_guard('D2', 'stages/stage_x11_qc.json'):
        q = nb.load_qc('stages/stage_x11_qc.json', RUN_ID)
        print('canónico:', q['canonical_method'])
        print(f"λ: Δλ={q['wavelength']['dlambda_A']:.3f} Å, marco={q['wavelength']['frame_final']}, {q['wavelength']['status']}")
        print(f"flujo: escala={q['flux']['scale_factor']:.3f} ({q['flux']['source'][:60]}...)")
        if q['flux'].get('declared_err_frac'):
            print(f"       flujo-cal declarado (NO en el total): {100 * q['flux']['declared_err_frac']:.1f}%"
                  f" — {q['flux']['declared_source'][:70]}")
        # La unidad que resolverán E3/G3: knob -> BUNIT del producto -> QC de M3.
        un = q['flux'].get('unit')
        if un:
            print(f"       unidad: {un['cgs']} erg/s/cm²/Å vía {un['source']}"
                  f" (BUNIT={un['from_bunit']}, M3={un['from_m3_qc']})")
            if un['conflict']:
                print('       ⚠ ' + un['conflict'])
        im = q['continuum']['intermethod_systematic']; ar = im['after_control_reference']
        print()
        print('continuo inter-método (psffit vs optimal_psfsub):')
        print(f"  ANTES  (crudo)       fraction_agree={im['fraction_channels_methods_agree']:.3f}  red_ratio={im['red_band_median_ratio']:.2f}×")
        print(f"  DESPUÉS (referenciado) fraction_agree={ar['fraction_channels_methods_agree']:.3f}  red_ratio={ar['red_band_median_ratio']:.2f}×")
        print(f"  sesgo rojo psffit/psfsub = {ar['canonical_control_bias_red']:+.0f} / {ar['other_control_bias_red']:+.0f}")
        v3 = q['checks']['v3_continuum_stable']
        print(f"\ncontinuo estable (V3): ok={v3['ok']} — {v3['fraction_channels_methods_agree']:.3f}"
              f" de los canales tienen los dos métodos de acuerdo dentro del error combinado,"
              f" umbral {v3['threshold']}")
        print(f"  métrica={v3['metric']}")
        print('  significa: lo que NO concuerda no puede ser el compañero (los dos métodos miden el mismo objeto)')
        print('             sino residuo cromático de la sustracción de halo -> afecta a la FORMA del continuo (G3),')
        print('             no a la línea Hα ni al límite de Ṁ (E1/E3).')

        # Los espectros definitivos: la tabla que D2 deja en su QC.
        sp = q.get('spectra')
        if sp is None:
            print('\n(este QC es anterior a la tabla de espectros: re-ejecuta D2 para tenerla)')
        else:
            n = lambda v, f='{:.1f}': '—' if v is None else f.format(v)
            print(f"\nespectros definitivos: {sp['n_companion']} métodos + {sp['n_primary']} primaria"
                  f" | unidad {sp['unit']} (consistente={sp['unit_consistent']})"
                  f" | misma rejilla={sp['companions_share_grid']}")
            print(f"medianas en {sp['red_band_A'][0]:.0f}–{sp['red_band_A'][1]:.0f} Å (donde el compañero se detecta)")
            print(f"  {'espectro':16s} {'papel':10s} {'S/N':>7s} {'flujo':>10s} {'err total':>10s} {'ratio/canon':>12s}")
            for row in sp['table']:
                tag = row['name'] + (' *' if row['canonical'] else '')
                print(f"  {tag:16s} {row['role']:10s} {n(row['red_snr_median']):>7s}"
                      f" {n(row['red_flux_median']):>10s} {n(row['flux_err_total_median']):>10s}"
                      f" {n(row['ratio_to_canonical_red'], '{:.2f}×'):>12s}")
            for row in sp['table']:
                if row['caveat']:
                    print(f"  ! {row['name']}: {row['caveat']}")


## Plot 1 — ¿el continuo del compañero es suyo o es halo de la primaria?

### La pregunta física

El compañero está a **1.18″** de una estrella unas **10³ veces más brillante** (en el rojo; más aún en el azul), y a esa separación seguimos dentro de su halo AO. Todo espectro del compañero es, por tanto, un **residuo**: lo que queda tras restar el halo. Cada método de extracción lo resta de una forma distinta, así que la pregunta obligada antes de creerse la forma del continuo es: *¿esta pendiente y estas bandas son del compañero, o son lo que a este método le sobró del halo?*

### Cómo se responde (y qué es «V3»)

**V3** es la tercera verificación de la spec de D2 (§6), y «gate» / «gatea» quiere decir que es un chequeo con umbral cuyo resultado entra en el semáforo final de F1 — no un cálculo que cambie ningún dato. En el QC vive como `checks.v3_continuum_stable`.

La idea: **dos métodos independientes miden el mismo objeto real**, así que su continuo debería coincidir *dentro del error*. Lo que no coincide no puede ser el compañero — es sistemático del método. Se mide como la **fracción de canales** en los que

> |continuo_A − continuo_B| ≤ √(σ_A² + σ_B²)  (errores estadísticos combinados)

y se exige ≥ **0.9**. El par no es fijo: lo eligen D1/D2 y lo declara el QC — aquí es **psffit** (canónico) vs **optimal_psfsub**.

### Por qué «referenciado a controles» (los dos paneles)

Además del residuo en el compañero, cada método deja un **pedestal en sitios donde no hay nada** — controles al mismo radio, procesados igual. Ese pedestal es común y no dice nada del método: compararlo cuenta el halo dos veces. Restando a cada método su propio nivel de control queda solo la discrepancia **genuinamente dependiente del método**. Izquierda = crudos; derecha = referenciados.

Para este objeto: 0.692 (crudo) → 0.579 (referenciado). Referenciar **no siempre sube** la cifra: acerca los continuos donde el pedestal era común y puede bajarla si el residuo cromático no lo es — eso también es información.

### Qué significa que falle

Que parte de lo que llamamos continuo del compañero es **residuo cromático de la sustracción de halo**, no su SED. Consecuencia acotada:

- **Afecta a G3** (ajuste atmosférico, Teff / tipo espectral), que vive de la *forma* del continuo → por eso G3 usa `cont_runmed_biasref`.
- **No afecta a Hα** (E1) ni al límite de Ṁ (E3): la línea es estrecha y se mide contra su continuo local, con controles procesados igual que el objeto.

No es un defecto de PSF — C1 ya usa Psfao. El diagnóstico completo, con las tres causas reales, está en [`docs/d2_red_continuum_diagnosis.md`](../../docs/d2_red_continuum_diagnosis.md).

*(La métrica original de la spec era |runmed − poly| sobre un solo espectro; se sustituyó porque confunde la estructura molecular **real** de una enana fría con el sistemático. Se conserva como `legacy_runmed_poly_fraction`.)*

*(En el azul, λ<7000 Å, el compañero tiene SNR<1: la discrepancia de esa zona está dentro del error combinado y no es señal.)*


In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt
    from astropy.io import fits
    rd = nb.run_dir(RUN_ID)
    qx = nb.load_qc('stages/stage_x11_qc.json', RUN_ID)
    im = qx['continuum']['intermethod_systematic']; ar = im['after_control_reference']
    # El par lo declara el QC (D1/D2 lo eligen: el primer par primario que
    # contiene al canónico, excluyendo sgf). Fijarlo a mano mostraba una
    # comparación distinta de la que gatea v3 en cuanto cambiaba de objeto.
    pair = [qx['canonical_method'], im['canonical_vs']]
    def col(method, c):
        h = fits.open(rd / 'stages' / f'spec_calibrated_{method}_object.fits')
        v = np.asarray(h[1].data[c], float); h.close(); return v
    wave = col(pair[0], 'wave_A')   # eje λ del producto
    fig, (axl, axr) = plt.subplots(1, 2, figsize=(13, 4.3), sharey=True)
    panels = ((axl, 'cont_runmed', 'ANTES: continuos crudos', im['fraction_channels_methods_agree']),
              (axr, 'cont_runmed_biasref', 'DESPUÉS: referenciados a controles', ar['fraction_channels_methods_agree']))
    drawn = []
    for ax, c, title, frac in panels:
        for method, color in zip(pair, ('tab:blue', 'tab:orange')):
            y = col(method, c); drawn.append(y)
            ax.plot(wave, y, lw=1.1, color=color,
                    label=method + (' (canónico)' if method == pair[0] else ''))
        ax.set_title(f'{title}\n{frac:.3f} de canales dentro del error combinado', fontsize=10)
    for ax in (axl, axr):
        ax.set_xlabel('λ [Å]'); ax.axvline(6563, color='tab:red', ls=':'); ax.axhline(0, color='0.7', lw=0.6); ax.legend(fontsize=8)
    axl.set_ylabel('continuo')
    # Límites por percentil: la escala fija estaba dimensionada para el flujo
    # del primer objeto y recortaba el continuo de cualquier otro.
    vals = np.concatenate([y[np.isfinite(y)] for y in drawn])
    if vals.size:
        lo, hi = np.percentile(vals, [1, 99]); pad = 0.15 * (hi - lo) or 1.0
        axl.set_ylim(lo - pad, hi + pad)
    axl.text(0.02, 0.04, 'azul (λ<7000): SNR<1\n(dentro del error)', fontsize=7,
             color='0.4', transform=axl.transAxes)
    fig.suptitle(f"D2 · ¿el continuo es del compañero o es halo residual?"
                 f"  ·  {pair[0]} vs {pair[1]}  ·  chequeo V3 de continuo estable")
    fig.tight_layout()
    outdir = rd / 'plots' / 'd2_calibrate'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'referencing.png', dpi=110); print('figura ->', outdir / 'referencing.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Plot 2 — el presupuesto de error

Cada componente del error del `spec_final_object.fits` vs λ (escala log). El total está **dominado por el `stat`** (empírico, M5 rojo); el sistemático de continuo (runmed vs poly) es el segundo; flujo-cal/cielo/telúrico son ~0.

**Las líneas discontinuas NO entran en `flux_err_total`** (la leyenda lo repite en cada una), y son dos casos distintos:

- `sys_continuum` — por diseño de la spec (§3.5): el continuo se entrega como columna y es el modelado quien decide si lo usa; sumarlo aquí sería decidir por él.
- `sys_fluxcal_declared` — el desvío de calibración absoluta que M3 mide frente a Gaia (|1−`flux_factor`|), **declarado y no plegado**: M3 no publica barra de error, y plegarlo movería el error del compañero, que sostiene decisiones congeladas. G3 ya asume su propio 10% (`g3_sys_fluxcal_frac`), mayor que este valor.

El total sólido es, por tanto, `stat ⊕ fluxcal ⊕ psf ⊕ cielo ⊕ telúrico`.


In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt
    from astropy.io import fits
    rd = nb.run_dir(RUN_ID)
    h = fits.open(rd / 'stages' / 'spec_final_object.fits'); d = h[1].data
    wave = np.asarray(d['wave_A'], float)   # eje λ del propio producto
    # `sumado` = si el termino entra en flux_err_total (spec 3.5). El continuo
    # NUNCA entro: se entrega como columna. Se dibujan discontinuos los dos que
    # no suman, para no leerlos como parte del total.
    comp = [('stat', 'flux_err_stat', True), ('flujo-cal', 'sys_fluxcal', True),
            ('psf', 'sys_psf', True), ('cielo', 'sys_sky', True),
            ('telúrico', 'sys_telluric', True), ('continuo', 'sys_continuum', False),
            ('flujo-cal DECLARADO', 'sys_fluxcal_declared', False)]
    sm = lambda x, n=51: np.convolve(np.nan_to_num(np.abs(x)), np.ones(n) / n, mode='same')
    fig, ax = plt.subplots(figsize=(11, 4))
    for lab, c, summed in comp:
        if c in d.columns.names:
            ax.plot(wave, sm(np.asarray(d[c], float)), lw=1 if summed else 1.3,
                    ls='-' if summed else '--',
                    label=lab if summed else f'{lab} (NO en el total)')
    ax.plot(wave, sm(np.asarray(d['flux_err_total'], float)), lw=2, color='k',
            label='TOTAL = stat ⊕ fluxcal ⊕ psf ⊕ cielo ⊕ telúrico')
    h.close()
    ax.set_yscale('log'); ax.set_xlabel('λ [Å]'); ax.set_ylabel('error (|componente|, suavizado)')
    ax.set_title('D2 · presupuesto de error: continuo y flujo-cal declarado quedan FUERA del total')
    ax.legend(fontsize=7, ncol=3)
    outdir = rd / 'plots' / 'd2_calibrate'; outdir.mkdir(parents=True, exist_ok=True)
    fig.tight_layout(); fig.savefig(outdir / 'error_budget.png', dpi=110)
    print('figura ->', outdir / 'error_budget.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Plot 3 — los espectros definitivos (6 métodos + la primaria)

El entregable de D2 en una figura, dibujada con la **misma función que usa la etapa** (`musepipe.stages.definitive_spectra_figure`), leyendo los productos calibrados del run:

- **arriba:** la **primaria** con su banda de error total (stat + sistemáticos). Está ~1e3–1e4 veces por encima del compañero, así que necesita su propio panel.
- **centro:** el **compañero por los 6 métodos**, suavizado 15 canales para que se lean a la vez, sobre la banda de error total del canónico (sin suavizar). Hα en rojo punteado.
- **abajo:** el **continuo referenciado a controles** — la comparación inter-método real (la del gate v3); la banda sombreada es el rojo, donde el compañero se detecta.

*(`sgf` filtra el continuo por construcción: su nivel no es comparable, su valor está en la línea.)*


In [ ]:
try:
    import matplotlib.pyplot as plt
    from musepipe.stages import definitive_spectra_figure
    rd = nb.run_dir(RUN_ID)
    fig, axes = definitive_spectra_figure(rd / 'stages', plt=plt)
    outdir = rd / 'plots' / 'd2_calibrate'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'definitive_spectra.png', dpi=110)
    print('figura ->', outdir / 'definitive_spectra.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Decisiones y notas
- **Los espectros definitivos (6 métodos + primaria) son el entregable de D2**, antes del estudio de Hα: todos con `BUNIT` declarado y error total; la tabla vive en `qc['spectra']` y la figura en `plots/stage_x11_spectra.png`. · [`docs/espectros_definitivos_handoff_2026-07-25.md`](../docs/espectros_definitivos_handoff_2026-07-25.md)
- **Diagnóstico honesto del 'continuo rojo inestable'**: 3 cosas reales (señal de enana fría + sistemático de nivel inter-método + rigidez del polinomio). NO es defecto de PSF. · [`docs/d2_red_continuum_diagnosis.md`](../docs/d2_red_continuum_diagnosis.md)
- **Referenciación a controles integrada** (2026-07-11): el chequeo de continuo estable (V3) gatea sobre la métrica referenciada — a cada método se le resta su propio nivel en controles, para no contar dos veces el pedestal de halo que ambos comparten. Columna `cont_runmed_biasref` entregada para G3.
- D1 **ya era control-centrado** (su veredicto refleja el sistemático genuino); esto solo puso a D2 al mismo nivel. Lo que V3 sigue midiendo por debajo del umbral es sistemático cromático **genuino** (no el pedestal): limitación aceptada en F1, que afecta a la forma del continuo (G3) y no al endpoint de Hα.
- El sistemático rojo NO afecta la línea Hα ni el límite de Ṁ; escala de flujo 1.0 validada vs Gaia; error total dominado por el stat.


## Conclusión (registrada)

**D2: producto final `spec_final_object` (psffit); Δλ −0.074 Å baricéntrico; flujo escala 1.0 (Gaia); error dominado por el stat.**

- **Fecha:** calibración D1 v2 realineado (2026-07-09); referenciación integrada 2026-07-11.
- **λ/flujo:** todos los factores trazables a A4 (M1 offset, M3 Gaia).
- **Continuo:** referenciación a controles integrada (a cada método se le resta su propio nivel en controles); columna `cont_runmed_biasref` para G3.
- **Continuo estable (V3):** 0.579 de canales concuerdan entre los dos métodos dentro del error combinado, umbral 0.9 → ok=**False**. Lo que no concuerda es sistemático cromático **genuino** de sustracción de halo, no el pedestal común: limitación aceptada, F1 sigue yellow.
- **Endpoint intacto:** eso afecta a la FORMA del continuo (G3); la línea Hα (E1) y el límite de Ṁ (E3) no dependen de ello.
- **Espectros definitivos:** los 6 métodos + la primaria, con unidad y error total (tabla en `qc['spectra']`, figura en `plots/stage_x11_spectra.png`) — resultado en sí mismos, entregados antes del estudio de Hα.
- **Unidad de flujo:** `qc['flux']['unit']` deja resuelta y contrastada la escala que usarán E3/G3 (knob → `BUNIT` → `m3_flux.flux_unit_cgs`). Si el `BUNIT` del producto y la unidad con la que M3 midió el factor no coinciden, sale como `open_issue`.
- **Downstream:** `spec_final_object` alimenta E1, E3 y G2.
